# Retirement choice speed benchmarks

**Paper**: Dobrescu and Shanker (2022), *A Fast Upper Envelope Scan Method for Discrete-Continuous Dynamic Programming*

This benchmarking notebook solves the discrete retirement choice model from [Iskhakov et al. (2017)](https://onlinelibrary.wiley.com/doi/abs/10.3982/QE643) using FUES.
The emphasis is on the geometry and performance of the upper-envelope step rather than on the general mechanics of solving the model, so we rely on pre-existing model files and solver routines.

> **Note on the model files**
>
> This notebook loads the retirement model from a set of declarative YAML files written in `dolo+`.
> In this declarative language, a **stage** can be thought of as just one self-contained Bellman operator of the dynamic problem, such as the worker's consumption choice, the retiree's consumption choice, or the labour-market decision. A full period is built by linking these stages together as a graph representing the choice structure of the model.
>
> The YAML format used here is called `dolo+`. It is simply a structured, machine-readable way of writing down the model's states, controls, transitions, and Bellman objects. You do **not** need to learn that syntax to follow this notebook. The point of this repository is not to teach `dolo+` terminology; it is to demonstrate the scaling properties of FUES.

**Upper-envelope methods compared:**

| This notebook | Paper name | Reference |
|---------------|------------|-----------|
| FUES | FUES | Dobrescu & Shanker (2022) |
| MSS | MSS (monotone segment selection) | Iskhakov et al. (2017), via [HARK](https://econ-ark.org) |
| LTM | LTM (local triangulation) | Druedahl (2021), via [ConSav](https://github.com/NumEconCopenhagen/ConsumptionSaving) |
| RFC | RFC (rooftop cut) | Dobrescu & Shanker (2024) |

Load the basic imports, file paths, and model solver. We refer to a model as a `nest`, which is essentially a directed graph built up from the stages.
The function `solve_nest` solves a model by running backward induction on the stages in the graph.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time, sys, os, yaml, tempfile
from pathlib import Path
from IPython.display import Image, Markdown, display

# ── Repo paths (robust: walk up until we find pyproject.toml) ──
_here = Path(os.path.abspath('')).resolve()
REPO = _here
for _ in range(10):
    if (REPO / 'pyproject.toml').exists():
        break
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'src'))

SYNTAX_DIR = REPO / 'examples' / 'retirement' / 'syntax'

# ── Imports ──
from examples.retirement.solve import solve_nest  # main solver of a model 
from examples.retirement.outputs import (
    get_policy, get_timing, get_solution_at_age,  # helper functions that unpack and extract from the solution dict
    euler, consumption_deviation,
    # Straight-forward plotters for this notebook (consistent Nord style)
    setup_nb_style,
    nb_plot_egrids, nb_plot_egm_interactive,
    nb_plot_cons_ages, nb_plot_scaling,
)

# ── Apply notebook style globally ──
setup_nb_style()
print(f'REPO: {REPO}')

## 1. Model

An agent lives for $T$ periods. Each period she holds assets $a \geq 0$ and makes two choices: a **discrete** choice (work or retire) and a **continuous** choice (how much to consume). Working yields wage income $y$ but costs disutility $\delta$; retirement is absorbing.

**Worker's Bellman equation:**

$$V_t^1(a) = \max_{d \in \{\text{work},\text{retire}\}} Q_t^d(a)$$

where

$$Q_t^{\text{work}}(a) = \max_c \left\{ \log(c) - \delta + \beta V_{t+1}^1\bigl((1+r)a + y - c\bigr) \right\}$$

$$Q_t^{\text{retire}}(a) = \max_c \left\{ \log(c) + \beta V_{t+1}^0\bigl((1+r)a - c\bigr) \right\}$$

**Retiree's Bellman equation:**

$$V_t^0(a) = \max_c \left\{ \log(c) + \beta V_{t+1}^0\bigl((1+r)a - c\bigr) \right\}$$

The key difficulty: $V_{t+1}^1$ is **not concave** because it is the upper envelope of concave functions, each conditional on a different sequence of future discrete choices. This is where FUES comes in.

### Stage decomposition

We can equivalently decompose the model into three stages per period:

1. **`labour_mkt_decision`** (branching) — pure discrete choice: $\max(Q^{\text{work}} - \delta,\; Q^{\text{retire}})$. Assets pass through unchanged.
2. **`work_cons`** — worker EGM: cash-on-hand $w = (1+r)a + y$, choose $c$, save $b = w - c$. FUES cleans the non-concave upper envelope.
3. **`retire_cons`** — retiree EGM: cash-on-hand $w_{\text{ret}} = (1+r)a_{\text{ret}}$, choose $c$, save $b_{\text{ret}} = w_{\text{ret}} - c$. Standard concave problem (no upper envelope needed).

The period has two entry points. Workers enter via $a$ (into the branching stage); retirees enter via $a_{\text{ret}}$ (directly into `retire_cons`). Both the retire branch and direct retiree entry feed into the same `retire_cons` stage:

<div markdown="0">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 680 280" style="max-width:680px;width:100%;">
  <defs>
    <style>
      .dag-edge   { fill: none; stroke: currentColor; stroke-width: 1.2px; stroke-linecap: round; }
      .dag-arrow  { fill: currentColor; }
      .dag-perch  { fill: var(--md-default-bg-color, #fff); stroke: currentColor; stroke-width: 1.2px; }
      .dag-stg    { fill: currentColor; stroke: none; }
      .dag-bound  { fill: none; stroke: currentColor; stroke-width: 0.8px; stroke-dasharray: 4 3; opacity: 0.25; }
      .dag-field  { font-family: Georgia, 'Times New Roman', serif; font-style: italic; font-size: 15px; fill: currentColor; }
      .dag-sub    { font-size: 10.5px; }
      .dag-ptag   { font-family: Georgia, 'Times New Roman', serif; font-style: italic; font-size: 9.5px; fill: currentColor; opacity: 0.40; }
      .dag-slabel { font-family: 'Inter', 'Helvetica Neue', sans-serif; font-size: 11px; fill: currentColor; }
      .dag-branch { font-family: 'Inter', 'Helvetica Neue', sans-serif; font-size: 10px; fill: currentColor; font-style: italic; }
      .dag-period { font-family: 'Inter', 'Helvetica Neue', sans-serif; font-size: 9.5px; fill: currentColor; opacity: 0.30; font-style: italic; }
      .dag-legend { font-family: 'Inter', 'Helvetica Neue', sans-serif; font-size: 8px; fill: currentColor; opacity: 0.45; }
    </style>
    <marker id="dag-ah" markerWidth="8" markerHeight="6" refX="7" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L8,3 L0,6 L1.5,3 Z" class="dag-arrow"/>
    </marker>
  </defs>
  <rect class="dag-bound" x="14" y="14" width="652" height="252" rx="8"/>
  <text class="dag-period" x="26" y="30">LifecyclePeriod</text>
  <circle class="dag-stg" cx="548" cy="27" r="3.5"/>
  <text class="dag-legend" x="555" y="30">stage</text>
  <circle class="dag-perch" cx="600" cy="27" r="3.5"/>
  <text class="dag-legend" x="607" y="30">perch field</text>
  <circle class="dag-perch" cx="68" cy="105" r="6"/>
  <text class="dag-field" x="68" y="88" text-anchor="middle">a</text>
  <circle class="dag-perch" cx="68" cy="210" r="6"/>
  <text class="dag-field" x="68" y="198" text-anchor="middle">a<tspan class="dag-sub" dy="3">ret</tspan></text>
  <circle class="dag-stg" cx="220" cy="105" r="7"/>
  <text class="dag-slabel" x="220" y="82" text-anchor="middle">labour_mkt_decision</text>
  <circle class="dag-stg" cx="415" cy="65" r="7"/>
  <text class="dag-slabel" x="415" y="48" text-anchor="middle">work_cons</text>
  <circle class="dag-stg" cx="415" cy="192" r="7"/>
  <text class="dag-slabel" x="415" y="222" text-anchor="middle">retire_cons</text>
  <circle class="dag-perch" cx="575" cy="65" r="6"/>
  <text class="dag-field" x="575" y="48" text-anchor="middle">b</text>
  <circle class="dag-perch" cx="575" cy="192" r="6"/>
  <text class="dag-field" x="575" y="180" text-anchor="middle">b<tspan class="dag-sub" dy="3">ret</tspan></text>
  <line class="dag-edge" x1="75" y1="105" x2="212" y2="105" marker-end="url(#dag-ah)"/>
  <path class="dag-edge" d="M228,102 C265,82 360,65 407,65" marker-end="url(#dag-ah)"/>
  <text class="dag-branch" x="310" y="72">work</text>
  <path class="dag-edge" d="M228,109 C268,140 355,182 407,192" marker-end="url(#dag-ah)"/>
  <text class="dag-branch" x="298" y="165">retire</text>
  <path class="dag-edge" d="M75,210 C185,210 335,198 407,193" marker-end="url(#dag-ah)"/>
  <line class="dag-edge" x1="423" y1="65" x2="568" y2="65" marker-end="url(#dag-ah)"/>
  <line class="dag-edge" x1="423" y1="192" x2="568" y2="192" marker-end="url(#dag-ah)"/>
</svg>
</div>

The inter-period connector assigns $b \to a$ (worker) and $b_{\text{ret}} \to a_{\text{ret}}$ (retiree), making retirement structurally absorbing. 

Let us load the stage syntax templates from the `syntax` directory for each of the stages above.

In [ ]:
# ── Display the stage YAML declarations ──
stage_names = ['work_cons', 'retire_cons', 'labour_mkt_decision']

for name in stage_names:
    path = SYNTAX_DIR / 'stages' / name / f'{name}.yaml'
    print(f'\n{"=" * 60}')
    print(f'  {name}  ({path.relative_to(REPO)})')
    print(f'{"=" * 60}')
    print(path.read_text())

## 2. Calibration and settings

No parameters are hard-coded in Python. Let us load the baseline calibration and settings from the `syntax` directory to see what they look like.

In [ ]:
# ── Display calibration and settings ──
for fname in ['calibration.yaml', 'settings.yaml']:
    path = SYNTAX_DIR / fname
    print(f'\n--- {fname} ---')
    print(path.read_text())

## 3. Solve via the canonical pipeline

`solve_nest` is the single entry point. It loads the YAML files, applies the three functors (methodize, configure, calibrate) to each stage, builds the model, and runs backward induction according to the stage graph above.
We can override the baseline settings and calibrations by passing `calib_overrides` and `config_overrides` to `solve_nest`. We can also select the upper-envelope method via `method='FUES'`, `'DCEGM'`, `'RFC'`, or `'CONSAV'`.

Note: the `dolo+` declarative syntax is solver-agnostic — `solve_nest` is built as part of this repo, not part of the `dolo+` implementation.

In [ ]:
# ── Warmup (JIT compile) then timed solve ──
# Solve for 50 periods (longer horizon shows more kinks)
solve_nest(SYNTAX_DIR, method='FUES', config_overrides={'T': 50})

nest, model, stage_ops = solve_nest(
    SYNTAX_DIR, method='FUES', config_overrides={'T': 50})

print(f'Model: T={model.T}, grid_size={model.grid_size}, '
      f'beta={model.beta}, delta={model.delta}, y={model.y}')
print(f'Euler error: {euler(model, get_policy(nest, "c")):.6f}')

# ── Override: lower beta and increase grid size ──
nest2, model2, _ = solve_nest(
    SYNTAX_DIR, method='FUES',
    calib_overrides={'beta': 0.91},
    config_overrides={'grid_size': 3000, 'T': 50})

print(f'\nWith overrides: beta={model2.beta}, grid_size={model2.grid_size}')
print(f'Euler error: {euler(model2, get_policy(nest2, "c")):.6f}')

### 3.1 EGM grid plots

The plots below show the raw EGM correspondence (red circles), FUES-refined points (blue crosses), the value function line (black), and estimated intersection points (green stars). We plot the solution from the second solve above (lower $\beta$).

In [ ]:
# ── EGM grid with FUES refinement (static, notebook style) ──
plot_age = 37
fig = nb_plot_egrids(nest2, model2, age=plot_age)
plt.show()

**Interactive plots**

In [ ]:
# ── Interactive EGM grid plot (auto-centered on crossings, auto y-range) ──
from examples.retirement.outputs import nb_plot_egm_interactive

fig = nb_plot_egm_interactive(nest2, model2, age=plot_age, pad=10)
fig.show()

### 3.2 Consumption policy across ages

`nb_plot_cons_ages` shows consumption as a function of assets at three ages. The discontinuous jumps correspond to the retirement decision — agents with wealth above the threshold retire.

In [ ]:
# ── Consumption policy at multiple ages ──
from examples.retirement.outputs import nb_plot_cons_ages

fig = nb_plot_cons_ages(nest, model, ages=(20, 30, 40))
plt.show()

## 4. Method comparison

At this calibration, all four upper-envelope methods deliver essentially the same solution quality: the reported Euler errors are extremely close, and the small differences are numerical rather than economically meaningful. The substantive comparison is therefore computation speed.

In [ ]:
# ── Solve with all four methods (T=50) ──
methods = ['FUES', 'DCEGM', 'RFC', 'CONSAV']
# Display labels (API uses DCEGM/CONSAV internally)
_DISPLAY = {'FUES': 'FUES', 'DCEGM': 'MSS', 'RFC': 'RFC', 'CONSAV': 'LTM'}
results = {}

for method in methods:
    solve_nest(SYNTAX_DIR, method=method, config_overrides={'T': 50})  # warmup
    nest_m, model_m, _ = solve_nest(
        SYNTAX_DIR, method=method, config_overrides={'T': 50})
    c_pol = get_policy(nest_m, 'c')
    ue_t, solve_t = get_timing(nest_m)
    results[method] = {
        'euler': euler(model_m, c_pol),
        'ue_ms': ue_t * 1000,
        'total_ms': solve_t * 1000,
    }

# ── Results table ──
print(f'{"Method":>8s} | {"Euler Error":>12s} | {"UE (ms)":>10s} | {"Total (ms)":>12s}')
print('-' * 50)
for m in methods:
    r = results[m]
    print(f'{_DISPLAY[m]:>8s} | {r["euler"]:>12.6f} | {r["ue_ms"]:>10.3f} | {r["total_ms"]:>12.3f}')

## 5. Scaling: upper envelope time vs grid size

We sweep grid sizes from 1,000 to 15,000 and compare upper-envelope time across methods. FUES remains the fastest method throughout, with a much smaller constant factor than the alternatives.

MSS and RFC both scale approximately linearly in this experiment and stay relatively close to one another. LTM is competitive only on the very smallest grids, but its cost rises much more quickly and it is overtaken by the time we reach the medium-sized grids used in the main comparison.

In [ ]:
# ── Scaling sweep (equal steps for visual linearity check) ──
import gc

grid_sizes = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000, 13000, 14000, 15000]
scaling = {m: [] for m in methods}

for gs in grid_sizes:
    for method in methods:
        n, _, _ = solve_nest(
            SYNTAX_DIR, method=method,
            config_overrides={'grid_size': gs, 'T': 50})
        ue, _ = get_timing(n)
        scaling[method].append(ue * 1000)
        del n
    gc.collect()
    print(f'grid_size={gs:>6d}  '
          + '  '.join(f'{m}: {scaling[m][-1]:.2f}ms' for m in methods))

In [ ]:
# ── Scaling plot ──
from examples.retirement.outputs import nb_plot_scaling

fig = nb_plot_scaling(grid_sizes, scaling, methods=methods)
plt.show()

checkpoints = [1000, 3000, 12000]
idx_by_grid = {gs: i for i, gs in enumerate(grid_sizes)}

lines = [
    "**Notebook scaling observations (live):**",
    "",
    "The table below reports the upper-envelope time ratio `other method / FUES`, so larger numbers mean a larger speed advantage for FUES.",
    "",
    "| Grid size | MSS / FUES | RFC / FUES | LTM / FUES |",
    "|-----------|------------|------------|------------|",
]

for gs in checkpoints:
    i = idx_by_grid[gs]
    fues_t = scaling['FUES'][i]
    mss_ratio = scaling['DCEGM'][i] / fues_t
    rfc_ratio = scaling['RFC'][i] / fues_t
    ltm_ratio = scaling['CONSAV'][i] / fues_t
    lines.append(f"| {gs:,} | {mss_ratio:.1f}x | {rfc_ratio:.1f}x | {ltm_ratio:.1f}x |")

lines.extend([
    "",
    "MSS and RFC scale linearly and remain close to one another across the sweep, while LTM scales quadratically and becomes much more expensive as the grid grows. FUES scales less than linearly",
])

display(Markdown("\n".join(lines)))

Laptop timings can be noisy due to thermal throttling, OS scheduling, and background processes. The PBS cluster results below provide cleaner measurements on dedicated hardware.

## 6. PBS cluster timings (Gadi)

Compare the notebook scaling sweep (above) with timings from a PBS batch run on NCI Gadi.
The table below is parsed from a timing markdown file produced by `run.py --run-timings`.
We average UE time across all $\delta$ values for each grid size and method.

In [ ]:
# ── PBS cluster timings ──
from nb_pbs import plot_pbs_scaling

timing_path = REPO / 'examples' / 'retirement' / 'results' / \
    'retirement_20260305_125534' / 'retirement_timing.md'

fig = plot_pbs_scaling(timing_path)
plt.show()
